## 5. Segmentation via TensorFlow U-Net
Next, we segment the pits using a pre-trained U-Net model. 

*Note: The model expects a custom loss function (`dice_focal_loss`) to be passed during loading, even if we only run inference. We provide a dummy function to satisfy the keras loader requirements.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gwyfile

afm_data = np.load("data_numpy/1nm_QD_AFMs_AIX6468_along_curved_edge_1um_100pN_18gain.0_00000/channel_00___0_data.npy")

FileNotFoundError: [Errno 2] No such file or directory: 'data_numpy/1nm_QD_AFMs_AIX6468_along_curved_edge_1um_100pN_18gain.0_00000.0_00000/channel_00___0_data.npy'

In [ ]:
import copy

# Copy the original topography object
topo_corrected = copy.deepcopy(topo)

# 1. Row alignment
topo_corrected.correct_median_diff()

# 2. Scar removal
topo_corrected = topo_corrected.filter_scars_removal(0.7, inline=False)

# 3. 2D Plane fit
topo_corrected = topo_corrected.corr_fit2d(inline=False)

# Extract the processed pixel data as a NumPy array for segmentation
afm_data = topo_corrected.pixels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters, measure
from tensorflow import keras

# Define custom loss function fallback required for model loading
def dice_focal_loss(y_true, y_pred): 
    return 0

# Model path
unet_path = r"segmentation_model.keras"

try:
    # Load model without compiling, injecting the custom loss object
    unet_model = keras.models.load_model(
        unet_path, 
        compile=False, 
        custom_objects={"dice_focal_loss": dice_focal_loss}
    )
    print("U-Net loaded successfully.")
    
    # Preprocess image: format shape to (1, H, W, 1) and cast to float32
    img_tensor = afm_data.astype(np.float32)[np.newaxis, ..., np.newaxis]
    
    # Run prediction
    pred_mask = unet_model.predict(img_tensor, verbose=0)
    
    # Threshold the sigmoid output to create a strict binary mask
    binary_mask_unet = (pred_mask[0, ..., 0] > 0.5).astype(np.uint8)
    
    # Visualize U-Net Results
    fig, (ax1, ax2) = plt.subplots(1, 2)
    ax1.imshow(afm_data, cmap='afmhot')
    ax1.set_title("Corrected AFM")
    ax1.axis('off')

    ax2.imshow(binary_mask_unet, cmap='gray')
    ax2.set_title("U-Net Segmentation")
    ax2.axis('off')
    plt.show()
    
except Exception as e:
    print(f"Failed to load or run U-Net. Ensure {unet_path} is in the current directory.")
    print(f"Error: {e}")